# SimCLR & contrastive learning (NT-Xent / InfoNCE)

_Semi & Self-Supervised Learning_

**Make two augmented views of the same image agree, and disagree with everything else in the batch. No labels needed.**

SimCLR (Simple framework for Contrastive Learning of visual Representations, Chen et al. 2020) learns useful image features with no labels at all.

       The trick: take one image, make two random "augmented views" of it (a crop, a color shift, a blur). Those two views are a positive pair — they show the same thing, so the network should map them to nearby points.

---

This notebook is a practice scaffold for the **SimCLR & contrastive learning (NT-Xent / InfoNCE)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — The NT-Xent / InfoNCE contrastive loss

This is the heart of SimCLR. Given the projected vectors of two views of each image, we L2-normalize them so dot products become cosine similarities, then divide by a temperature `tau`. We mask the diagonal so no view matches itself, point each view at its true partner (the same image's other view sits `N` rows away), and apply a softmax cross-entropy that must pick the real partner out of all `2N-1` other views.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18


def nt_xent_loss(z1, z2, tau=0.5):
    """NT-Xent / InfoNCE over a batch of paired views.
    z1, z2: (N, d) projections of view-1 and view-2 of the same N images."""
    N = z1.size(0)
    z = F.normalize(torch.cat([z1, z2], dim=0), dim=1)   # (2N, d) unit vectors
    sim = z @ z.t() / tau                                 # (2N, 2N) cosine sims / tau

    # mask out self-similarity on the diagonal so a view never matches itself
    self_mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim.masked_fill_(self_mask, float('-inf'))

    # for view i, its positive partner sits N positions away (and vice versa)
    targets = torch.arange(2 * N, device=z.device)
    targets = (targets + N) % (2 * N)

    # softmax cross-entropy: pick the true partner out of all 2N-1 other views
    return F.cross_entropy(sim, targets)

### Step 2 — The SimCLR network: encoder plus projection head

SimCLR has two parts. An **encoder** `f` (here a ResNet-18 with its classifier removed) maps an image to a representation `h` — this is the part we keep for downstream tasks. A small **projection head** `g` maps `h` to `z`, and the contrastive loss is applied on `z`, not `h`. The projection head is discarded after pretraining.

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        backbone = resnet18(weights=None)
        feat_dim = backbone.fc.in_features          # 512 for resnet18
        backbone.fc = nn.Identity()                 # encoder f: image -> h
        self.encoder = backbone
        self.projector = nn.Sequential(             # projection head g: h -> z
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim))

    def forward(self, x):
        h = self.encoder(x)                         # representation to KEEP downstream
        z = self.projector(h)                       # projection used ONLY for the loss
        return h, z

### Step 3 — One SimCLR training step

Each step makes two independent **augmented views** of the same batch of images, runs both through the network, and keeps only the projections `z1`, `z2` (we discard `h` during pretraining). The NT-Xent loss then pulls each pair of views together and pushes everything else apart. After pretraining you throw away the projector and keep the encoder for linear probing or fine-tuning on a small labeled set.

In [ ]:
# ---- one SimCLR training step --------------------------------------------
# 'augment' is a strong random transform (RandomResizedCrop + color jitter + blur).
model = SimCLR().cuda()
opt = torch.optim.Adam(model.parameters(), lr=3e-4)


def train_step(images, augment, tau=0.5):
    # two independent augmented views of the SAME batch of images
    x1 = augment(images).cuda()
    x2 = augment(images).cuda()

    _, z1 = model(x1)                               # discard h during pretraining
    _, z2 = model(x2)

    loss = nt_xent_loss(z1, z2, tau)
    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss.item()


# After pretraining: throw away model.projector, keep model.encoder (gives h),
# then linear-probe or fine-tune model.encoder on your small labeled set.

## Visualize the data & results

_Do augmentation-invariant features (a SimCLR-style proxy) beat raw pixels when you have only a handful of labels?_

### Step 1 — Load digits and define the augmentation

We use the 1797 real 8x8 handwritten digits with pixels scaled to 0..1. The `augment` function is a faithful tiny stand-in for SimCLR's augmentation pipeline: a small random shift plus Gaussian noise. Calling it twice on the same image produces two independent 'views' — exactly the positive pairs SimCLR learns to make agree.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# REAL data: 1797 handwritten 8x8 digits.
digits = load_digits()
X = digits.data / 16.0           # pixels in 0..1
y = digits.target
rng = np.random.RandomState(0)


def augment(imgs, r):
    """A faithful tiny stand-in for SimCLR's augmentation pipeline:
    small random shift + Gaussian noise. Two calls = two 'views'."""
    out = []
    for v in imgs:
        img = v.reshape(8, 8)
        img = np.roll(img, r.randint(-1, 2), axis=0)   # shift up/down
        img = np.roll(img, r.randint(-1, 2), axis=1)   # shift left/right
        noisy = np.clip(img + 0.07 * r.randn(8, 8), 0, 1)
        out.append(noisy.reshape(-1))
    return np.array(out)

### Step 2 — Pretrain an augmentation-invariant encoder (no labels)

We take 800 images, **hide their class labels**, and build 4 augmented views of each. Real SimCLR trains a deep ResNet with NT-Xent on a GPU; here we fit a PCA on the augmented views as a faithful small CPU stand-in for an augmentation-invariant encoder — it learns directions robust to the augmentation noise. This whole step uses no labels at all.

In [ ]:
# Self-supervised pretraining pool: images whose CLASS LABELS WE HIDE.
n_base = 800
base_idx = rng.choice(len(X), n_base, replace=False)
base = X[base_idx]

# Build 4 augmented views per image (no labels used at all).
A = np.vstack([augment(base, rng) for _ in range(4)])

# 'Encoder' proxy: learn directions robust to the augmentation noise.
# (Real SimCLR uses a deep ResNet + NT-Xent on a GPU; PCA on augmented
#  views is a faithful small CPU stand-in for an augmentation-invariant encoder.)
encoder = PCA(n_components=16, random_state=0).fit(A)

### Step 3 — Probe with few labels: learned features vs raw pixels

Now we evaluate on held-out **real** images with their true labels. We compare two feature sets fed to a kNN: the encoder's learned representation versus the raw pixels. By sweeping the number of labeled training points, we see whether the augmentation-invariant features win when labels are scarce — the whole promise of self-supervised pretraining.

In [ ]:
# Downstream eval: held-out REAL images + their true class labels.
eval_idx = rng.choice(np.setdiff1d(np.arange(len(X)), base_idx), 900, replace=False)
Xe, ye = X[eval_idx], y[eval_idx]
Re = encoder.transform(Xe)        # learned representation
Pe = Xe                            # raw-pixel baseline


def knn_acc_vs_labels(feat, counts, trials=40):
    accs = []
    for n_lab in counts:
        s = []
        for t in range(trials):
            r = np.random.RandomState(7000 + t)
            perm = r.permutation(len(feat))
            tr, te = perm[:n_lab], perm[n_lab:n_lab + 300]
            if len(np.unique(ye[tr])) < 2:
                continue
            clf = KNeighborsClassifier(n_neighbors=min(3, n_lab)).fit(feat[tr], ye[tr])
            s.append(clf.score(feat[te], ye[te]))
        accs.append(round(float(np.mean(s)), 3))
    return accs


counts = [5, 10, 20, 40, 80, 160]
print("labels        ", counts)
print("learned rep   ", knn_acc_vs_labels(Re, counts))
print("raw pixels    ", knn_acc_vs_labels(Pe, counts))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In a SimCLR batch, exactly which other views are the positive for view $i$, and which are negatives? If the batch has $N = 64$ images, how many negatives does each view have?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how positives are made: two augmented views of the SAME image. — _The only positive of view $i$ is its augmented twin from the same original image._
- Count the total views: $2N = 128$. — _Each of the 64 images contributes two views._
- Subtract the view itself and its one positive partner: $128 - 1 - 1 = 126$. — _Every remaining view comes from a different image, so it is a negative._

**Answer:** The single positive is view $i$'s augmented twin (the other view of the same image). The $2N - 2 = 126$ views from the other 63 images are all negatives. So each view is contrasted against its one partner and 126 distractors.

</details>

**Problem 2.** A positive pair has $\text{sim}(z_i, z_j) = 0.6$. The two negatives have similarities $0.5$ and $0.4$. With temperature $\tau = 0.2$, find the NT-Xent loss $\ell_{i,j}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scale by $1/\tau = 5$ and exponentiate. Positive: $\exp(0.6 \times 5) = \exp(3.0) \approx 20.09$. — _Small $\tau$ sharply amplifies similarity gaps._
- Negatives: $\exp(0.5 \times 5) = \exp(2.5) \approx 12.18$ and $\exp(0.4 \times 5) = \exp(2.0) \approx 7.39$. — _Apply the same scaling to every candidate._
- Numerator $= 20.09$. Denominator $= 20.09 + 12.18 + 7.39 = 39.66$. — _The denominator sums the positive plus all negatives (skipping the view itself)._
- Probability $= 20.09 / 39.66 \approx 0.507$. Loss $= -\log(0.507) \approx 0.680$. — _The softmax fraction is the chance of picking the true partner; $-\log$ of it is the loss._

**Answer:** $\ell_{i,j} = -\log\!\frac{e^{3.0}}{e^{3.0} + e^{2.5} + e^{2.0}} = -\log(0.507) \approx 0.68$. The negatives at $0.5$ and $0.4$ are nearly as similar as the positive at $0.6$, so the loss stays high — these are hard negatives the encoder still has to separate.

</details>

**Problem 3.** Your teammate trains SimCLR, then attaches a classifier to the projected vector $z = g(h)$ and gets poor accuracy. What is the likely fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the two outputs: the encoder representation $h = f(\tilde{x})$ and the projected $z = g(h)$. — _The contrastive loss is applied on $z$, not $h$._
- Recall what $g$ does: it strips information not needed for the contrastive task. — _That stripped information is often exactly what a downstream classifier needs._
- Use $h$ (before the projection head) for downstream, and discard $g$. — _$h$ retains the richer, more transferable features._

**Answer:** Feed the classifier the encoder output $h$, not the projected $z$. SimCLR found that the layer before the projection head transfers far better, because the head discards detail the contrastive loss did not need. Keep $f$, throw away $g$.

</details>